# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata (does not download files yet)
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List RecordSets and their fields using @id
if not metadata.record_sets:
    print("No record sets found in metadata. The Croissant schema may list record sets inside file objects (e.g., CSV files).\nTrying to inspect the DataFileObjects.")
    # Let's list all DataFileObjects to identify record sets
    if hasattr(metadata, "data_files"):
        for file_obj in metadata.data_files:
            print(f"DataFileObject @id: {getattr(file_obj, '@id', None)}; name: {getattr(file_obj, 'name', None)}")
else:
    for rs in metadata.record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print("  Fields:")
        for f in rs['fields']:
            print(f"    - {f['@id']}: {f['name']}")

# Show how to enumerate records for the main record set (using @id)
print("\nExample: Listing records from the main record set (replace with detected record_set @id):")
# Try to autodetect record set id by loading records
try:
    for rec in dataset.records():
        pprint.pprint(rec)
        break  # Only display one example
except Exception as e:
    print(f"Could not enumerate records: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We can typically use the main record set (or the default if only one exists)
# You may specify a record_set @id if more than one exists; here assume None/auto
main_record_set_id = None  # Use None if there's only one record set

records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print("Columns in the extracted DataFrame:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field to analyze. Based on mass spectrometry convention, candidates may include 'MW' (molecular weight), 'CoveragePercent', 'NumPeptides', or normalized abundance columns.
numeric_field = 'MW' if 'MW' in df.columns else df.select_dtypes(include=[float, int]).columns[0]
print(f"Using numeric field: {numeric_field}\n")

threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records\n")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' (z-score) for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if present (e.g., 'Description' or 'Sample', etc.)
candidate_cats = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
group_field = None
for candidate in candidate_cats:
    if filtered_df[candidate].nunique() > 1 and filtered_df[candidate].nunique() < 20:
        group_field = candidate
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
    print(grouped_df.head())
else:
    print("\nNo suitable categorical field for grouping found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the main numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# Visualize grouped data if available
if group_field:
    plt.figure(figsize=(10,4))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the FAIR² mass spectrometry dataset of extracellular vesicles from stimulated human mast cells.

- We loaded metadata and inspected record sets and fields via their Croissant `@id`s.
- We extracted the main record set, explored numeric attributes such as molecular weight, filtered and normalized the data, and grouped by categories where possible.
- Visualizations revealed field distributions and group effects.

This workflow demonstrates how standardized Croissant schemas enable systematic, reproducible exploration and processing of FAIR biomedical datasets.